# 🧱 Module 5: Scaling with Mean Stacking

In Module 1, we looked at one image. In reality, astronomical signals (like Pulsars or distant Galaxies) are often buried in noise. To see them, we "stack" multiple images by calculating the mean of every pixel across many frames.

---

In [ ]:
import numpy as np
from astropy.io import fits
import matplotlib.pyplot as plt
from astropy.utils.data import download_file
import time

# Function to simulate fetching multiple frames of the same sky region
def get_sample_stack(n_frames=5):
    url = 'https://archive.stsci.edu/pub/vosamp/horsehead.fits'
    fits_file = download_file(url, cache=True)
    with fits.open(fits_file) as hdul:
        base_data = hdul[0].data.astype(np.float32)
    
    stack = []
    for i in range(n_frames):
        # Add artificial noise to simulate different exposures
        noise = np.random.normal(0, 50, base_data.shape)
        stack.append(base_data + noise)
    return np.array(stack)

print("🛰️ Simulating a stack of 10 astronomical exposures...")
image_stack = get_sample_stack(10)
print(f"📦 Stack Shape: {image_stack.shape} (Frames, Height, Width)")

## 1. Simple Mean Stacking
The simplest way to reduce noise is to take the mean.

In [ ]:
start = time.time()
mean_image = np.mean(image_stack, axis=0)
print(f"⏱️ Mean stacking took: {time.time() - start:.4f} seconds")

plt.figure(figsize=(15, 5))
plt.subplot(1, 2, 1)
plt.imshow(image_stack[0], cmap='gray', origin='lower', vmax=2000)
plt.title("Single Noisy Exposure")

plt.subplot(1, 2, 2)
plt.imshow(mean_image, cmap='gray', origin='lower', vmax=2000)
plt.title("Mean Stacked (10 Frames)")
plt.show()

## 2. The Median Problem
Mean stacking is fast, but it doesn't handle **outliers** (like satellite trails or cosmic rays) well. Median stacking is better but much slower because it requires sorting all pixels.

**Challenge**: Try running `np.median` on a very large stack and notice the performance hit. This is why we use distributed compute!